# SpikeData demo

Small walkthrough of the new `sim_data_analyzer.spike_data.SpikeData` API.

In [5]:
import pickle
import sys
from pathlib import Path

import numpy as np

DIR_PACKAGE = Path.cwd().parent
DIR_REPO = DIR_PACKAGE.parent
if str(DIR_REPO) not in sys.path:
    sys.path.insert(0, str(DIR_REPO))

from sim_data_analyzer.spike_data import SpikeData


In [6]:
# Parameters
FPATH_SIM_RESULT = (
    DIR_PACKAGE / 'dev_scratch' / 'data_src' / 'a1_lfp_30s' / 'data_00000_seed_1000.pkl'
)
DIRPATH_OUT = DIR_PACKAGE / 'dev_scratch' / 'data_proc' / 'a1_lfp_30s_0' / 'spike_data_demo'

T0 = 5.0
TMAX = 30.0
MS = True
NDIGITS = 3

FPATH_COMBINED = DIRPATH_OUT / 'spikes_combined_ms.npz'
FPATH_PER_CELL = DIRPATH_OUT / 'spikes_per_cell_ms.npz'

DIRPATH_OUT.mkdir(parents=True, exist_ok=True)
FPATH_SIM_RESULT


WindowsPath('d:/WORK/Salvador/repo/sim_data_analyzer/dev_scratch/data_src/a1_lfp_30s/data_00000_seed_1000.pkl')

In [7]:
# Load the NetPyNE simulation result
with FPATH_SIM_RESULT.open('rb') as fobj:
    sim_result = pickle.load(fobj)

sorted(sim_result['net']['pops'])[:10], len(sim_result['net']['pops'])


(['CT5A',
  'CT5Afrz',
  'CT5B',
  'CT5Bfrz',
  'CT6',
  'CT6frz',
  'HTC',
  'HTCfrz',
  'IRE',
  'IREM'],
 57)

In [8]:
# Extract combined spikes only
spikes_combined = SpikeData.from_sim_result(
    sim_result,
    combine=True,
    t0=T0,
    tmax=TMAX,
    subtract_t0=True,
    ms=MS,
    ndigits=NDIGITS,
    
)

print(spikes_combined.metadata)
print('First 8 populations:', spikes_combined.get_pop_names()[:8])
print('IT3 combined spikes:', spikes_combined.get_pop_spikes('IT3')[0][:20])


{'combine': True, 't0': 5.0, 'tmax': 30.0, 'subtract_t0': True, 'ms': True, 'ndigits': 3}
First 8 populations: ['IT2', 'IT3', 'ITP4', 'ITS4', 'IT5A', 'IT5B', 'IT6', 'CT5A']
IT3 combined spikes: [0.2  0.65 0.7  0.85 1.1  1.2  1.35 1.5  1.6  1.7  2.1  2.3  2.35 2.45
 2.45 2.7  2.8  2.85 2.85 2.95]


In [9]:
# Save/load the combined representation
spikes_combined.save(FPATH_COMBINED)
spikes_combined_loaded = SpikeData.load(FPATH_COMBINED)

print(FPATH_COMBINED)
print(spikes_combined_loaded.metadata)
np.array_equal(
    spikes_combined.get_pop_spikes('IT3')[0],
    spikes_combined_loaded.get_pop_spikes('IT3')[0],
)


d:\WORK\Salvador\repo\sim_data_analyzer\dev_scratch\data_proc\a1_lfp_30s_0\spike_data_demo\spikes_combined_ms.npz
{'combine': True, 't0': 5.0, 'tmax': 30.0, 'subtract_t0': True, 'ms': True, 'ndigits': 3}


True

In [12]:
S = spikes_combined.get_net_spikes()
S

{'IT2': [array([3.300000e+00, 4.050000e+00, 9.300000e+00, ..., 2.499495e+04,
         2.499640e+04, 2.499885e+04])],
 'IT3': [array([2.000000e-01, 6.500000e-01, 7.000000e-01, ..., 2.499985e+04,
         2.499990e+04, 2.499990e+04])],
 'ITP4': [array([1.500000e-01, 3.500000e-01, 8.500000e-01, ..., 2.499970e+04,
         2.499980e+04, 2.499995e+04])],
 'ITS4': [array([1.05000e+00, 1.20000e+00, 1.95000e+00, ..., 2.49995e+04,
         2.49995e+04, 2.49999e+04])],
 'IT5A': [array([2.20000e+00, 2.45000e+00, 4.30000e+00, ..., 2.49996e+04,
         2.49997e+04, 2.49999e+04])],
 'IT5B': [array([3.00000e-01, 4.00000e-01, 7.00000e-01, ..., 2.49981e+04,
         2.49982e+04, 2.49998e+04])],
 'IT6': [array([2.500000e-01, 6.000000e-01, 9.000000e-01, ..., 2.499910e+04,
         2.499985e+04, 2.499995e+04])],
 'CT5A': [array([2.00000e-01, 1.20000e+00, 1.90000e+00, ..., 2.49937e+04,
         2.49981e+04, 2.49982e+04])],
 'CT5B': [array([1.050000e+00, 2.350000e+00, 4.600000e+00, ..., 2.499755e+04,
     

In [10]:
# Extract per-cell spikes
spikes_per_cell = SpikeData.from_sim_result(
    sim_result,
    combine=False,
    t0=T0,
    tmax=TMAX,
    subtract_t0=True,
    ms=MS,
    ndigits=NDIGITS,
)

spikes_per_cell.save(FPATH_PER_CELL)
spikes_per_cell_loaded = SpikeData.load(FPATH_PER_CELL)

print(FPATH_PER_CELL)
print('IT3 cell gids:', spikes_per_cell_loaded.get_pop_cell_gids('IT3'))
print('IT3 number of cell trains:', len(spikes_per_cell_loaded.get_pop_spikes('IT3')))
print('IT3 first cell spikes:', spikes_per_cell_loaded.get_pop_spikes('IT3')[0][:20])


KeyboardInterrupt: 

In [ ]:
# Combine per-cell spikes after loading
spikes_recombined = spikes_per_cell_loaded.combine()

same_it3 = np.array_equal(
    spikes_recombined.get_pop_spikes('IT3')[0],
    spikes_combined.get_pop_spikes('IT3')[0],
)

print('Recombined IT3 matches direct combined extraction:', same_it3)
print('Recombined IT3 spikes:', spikes_recombined.get_pop_spikes('IT3')[0][:20])
